## 1. 使用滑动窗口生成数据

In [2]:
import re

In [3]:
text = "LLM learns from next token prediction but exhibit general intelligence"
text = re.split(" ", text)

seq_len = 4

for start in range(len(text)-seq_len):
    for i in range(1, seq_len+1):
        print("context: ", text[start: start+i], "======>", text[start+i])

context:  ['LLM'] ======> learns
context:  ['LLM', 'learns'] ======> from
context:  ['LLM', 'learns', 'from'] ======> next
context:  ['LLM', 'learns', 'from', 'next'] ======> token
context:  ['learns'] ======> from
context:  ['learns', 'from'] ======> next
context:  ['learns', 'from', 'next'] ======> token
context:  ['learns', 'from', 'next', 'token'] ======> prediction
context:  ['from'] ======> next
context:  ['from', 'next'] ======> token
context:  ['from', 'next', 'token'] ======> prediction
context:  ['from', 'next', 'token', 'prediction'] ======> but
context:  ['next'] ======> token
context:  ['next', 'token'] ======> prediction
context:  ['next', 'token', 'prediction'] ======> but
context:  ['next', 'token', 'prediction', 'but'] ======> exhibit
context:  ['token'] ======> prediction
context:  ['token', 'prediction'] ======> but
context:  ['token', 'prediction', 'but'] ======> exhibit
context:  ['token', 'prediction', 'but', 'exhibit'] ======> general
context:  ['prediction'] ===

## 2. 生成数据集Dataset

In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken

In [9]:
class GPTDataset(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.stride = stride

        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids)-max_length, stride):
            input_chunk = token_ids[i: i+max_length]
            target_chunk = token_ids[i+1: i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))


    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

## 3. 数据加载器Dataloader

In [12]:
with open("data/the-verdict.txt", "r") as f:
    text = f.read()


tokenizer = tiktoken.get_encoding("gpt2")
dataset = GPTDataset(text, tokenizer, max_length=16, stride=8)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, drop_last=True, num_workers=0)


In [13]:
len(dataset), dataset[0]

(642,
 (tensor([   40,   367,  2885,  1464,  1807,  3619,   402,   271, 10899,  2138,
            257,  7026, 15632,   438,  2016,   257]),
  tensor([  367,  2885,  1464,  1807,  3619,   402,   271, 10899,  2138,   257,
           7026, 15632,   438,  2016,   257,   922])))

In [11]:
# data_iter = iter(dataloader)
# input_ids, target_ids = next(data_iter)

i = 0
for input_ids, target_ids in dataloader:
    print("输入 token ids：", input_ids)
    print("目标 token ids：", target_ids)
    if i == 10:
        break
    i += 1

输入 token ids： tensor([[  319,   326,   966,   314,   714,   423,  1813,  4544,  9325,   701,
           262, 40830, 12719,  3874,    13,   632],
        [   11,  8759,  2763,    11,   373,   612,  1997,   319,  4534,   314,
          3636,   470,   423,  1813,   284,   423],
        [ 1092,   517,   621,   611,   314,  1549,  1239, 12615,   257, 14093,
           526,   198,   198,  1870,   465,  8216],
        [  607, 41793,    13,   632,   373,   465,   898, 41793,   339,  3947,
           284,   307,  1592,  2259,   739,   438]])
目标 token ids： tensor([[  326,   966,   314,   714,   423,  1813,  4544,  9325,   701,   262,
         40830, 12719,  3874,    13,   632,   373],
        [ 8759,  2763,    11,   373,   612,  1997,   319,  4534,   314,  3636,
           470,   423,  1813,   284,   423,   520],
        [  517,   621,   611,   314,  1549,  1239, 12615,   257, 14093,   526,
           198,   198,  1870,   465,  8216,  1297],
        [41793,    13,   632,   373,   465,   898, 417